In [1]:
import pandas as pd
import numpy as np

# 1. Load the cleaned dataset from Step 1
df = pd.read_csv('01_cleaned_data.csv')
print("Loaded Cleaned Dataset Shape:", df.shape)


Loaded Cleaned Dataset Shape: (100000, 25)


In [2]:
# 2. Temporal Features: Order Day, Day Type (Weekday vs Weekend) & Order Hour
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')
df['Order_Day_Name'] = df['Order_Date'].dt.day_name()
df['Order_Day_Type'] = np.where(df['Order_Date'].dt.dayofweek >= 5, 'Weekend', 'Weekday')

# Extract hour from Order_Time (handles HH:MM:SS or HH:MM formats)
df['Order_Hour'] = pd.to_datetime(df['Order_Time'].astype(str), errors='coerce').dt.hour.fillna(12).astype(int)

# Peak Hour Indicator (Lunch: 12 PM - 3 PM, Dinner: 7 PM - 10 PM)
# Combines existing Peak_Hour flag with standard meal rush hours
derived_peak = df['Order_Hour'].isin([12, 13, 14, 19, 20, 21])
df['Peak_Hour_Indicator'] = np.where(df['Peak_Hour'] | derived_peak, 'Peak', 'Non-Peak')


In [3]:
# 3. Financial & Profitability Features
# Profit Margin Percentage
df['Profit_Margin_Pct'] = (df['Profit_Margin'] * 100).round(2)

# Estimated Net Profit in currency units
df['Net_Profit'] = (df['Final_Amount'] * df['Profit_Margin']).round(2)

# Discount Percentage relative to Order Value
df['Discount_Pct'] = np.where(
    df['Order_Value'] > 0,
    ((df['Discount_Applied'] / df['Order_Value']) * 100).round(2),
    0.0
)


In [4]:
# 4. Delivery Performance & Customer Age Segmentation
# Delivery Performance Categories based on Delivery_Time_Min
df['Delivery_Performance'] = pd.cut(
    df['Delivery_Time_Min'],
    bins=[0, 30, 45, np.inf],
    labels=['Fast (<=30 min)', 'On-Time (31-45 min)', 'Delayed (>45 min)']
).astype(str)

# Customer Age Groups
df['Customer_Age_Group'] = pd.cut(
    df['Customer_Age'],
    bins=[0, 25, 35, 50, 120],
    labels=['18-25 (Gen Z)', '26-35 (Young Adults)', '36-50 (Mid-Career)', '50+ (Seniors)']
).astype(str)


In [6]:
# 5. Verify Engineered Features & Export Final Dataset
engineered_cols = [
    'Order_Day_Name', 'Order_Day_Type', 'Order_Hour', 'Peak_Hour_Indicator',
    'Profit_Margin_Pct', 'Net_Profit', 'Discount_Pct',
    'Delivery_Performance', 'Customer_Age_Group'
]

print("Engineered Features Preview:")
display(df[engineered_cols].head())

print("\nFinal Dataset Shape:", df.shape)
df.to_csv('03_final_data.csv', index=False)
print("Successfully saved 03_final_data.csv!")


Engineered Features Preview:


,Order_Day_Name,Order_Day_Type,Order_Hour,Peak_Hour_Indicator,Profit_Margin_Pct,Net_Profit,Discount_Pct,Delivery_Performance,Customer_Age_Group
0,Sunday,Weekend,0,Peak,13.0,149.11,4.18,Delayed (>45 min),18-25 (Gen Z)
1,Monday,Weekday,0,Peak,48.0,2327.52,0.41,Fast (<=30 min),36-50 (Mid-Career)
2,Sunday,Weekend,0,Peak,8.0,58.96,2.64,Delayed (>45 min),36-50 (Mid-Career)
3,Tuesday,Weekday,0,Non-Peak,4.0,45.88,4.18,Delayed (>45 min),36-50 (Mid-Career)
4,Sunday,Weekend,0,Non-Peak,12.0,42.24,5.38,Delayed (>45 min),50+ (Seniors)



Final Dataset Shape: (100000, 34)
Successfully saved 03_final_data.csv!
